# CV16 — Học quên có đối chứng

Bắt đầu từ source đã train, chia retain/forget và untouched eval. Mục tiêu minh họa giữ retain CE, gần reference trên retain, đẩy output forget về uniform. Không chứng minh xóa dữ liệu hay privacy.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Bắt đầu từ source đã train, chia retain/forget và untouched eval. Mục tiêu minh họa giữ retain CE, gần reference trên retain, đẩy output forget về uniform. Không chứng minh xóa dữ liệu hay privacy. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from copy import deepcopy
from vision.curriculum.cv16_unlearning import objective
retain_images = torch.rand(2, 3, 32, 32, device=device)
retain_labels = torch.tensor([0, 1], device=device)
forget_images = torch.rand(2, 3, 32, 32, device=device)
forget_labels = torch.tensor([1, 0], device=device)
model = nn.Sequential(nn.Flatten(), nn.Linear(3072, 2)).to(device)
reference = deepcopy(model).eval().requires_grad_(False)
batch = (retain_images, retain_labels, forget_images, forget_labels)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
with torch.no_grad():
    source_probabilities = reference(forget_images).softmax(-1)
print('Initial forget probabilities:', source_probabilities.tolist())
print('Reference stays fixed:', all(not p.requires_grad for p in reference.parameters()))


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = objective(model, reference, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Unchanged source → retain-only fine-tune → retain/reference/uniform objective → so với retrain retain-only. forget_y đang không dùng trong objective này, có chủ đích.


In [ ]:
model.eval()
with torch.no_grad():
    retain_accuracy = (model(retain_images).argmax(-1) == retain_labels).float().mean()
    forget_confidence = model(forget_images).softmax(-1).max(-1).values.mean()
print('Fixture retain accuracy:', retain_accuracy.item())
print('Fixture forget confidence:', forget_confidence.item())
print('Gold control: retrain from scratch on retain only; uniform predictions do not establish deletion.')


## Thử nghiệm tiếp theo

Unchanged source → retain-only fine-tune → retain/reference/uniform objective → so với retrain retain-only. forget_y đang không dùng trong objective này, có chủ đích.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv16_unlearning.py) và [tài liệu gốc](../REFERENCE.md).
